In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv('../data/cleaned/cleaned_superstore.csv')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Order Month No,Profit Margin
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,November,11,0.1600
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,November,11,0.3000
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,June,6,0.4700
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,October,10,-0.4000
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,October,10,0.1125


In [3]:
# Create connection
conn = sqlite3.connect('../data/cleaned/sales_database.db')

# Load dataframe into SQL table
df.to_sql('sales', conn, if_exists='replace', index=False)

9994

In [4]:
query = "SELECT COUNT(*) FROM sales;"
pd.read_sql(query, conn)

,COUNT(*)
0,9994


In [5]:
query = """
SELECT 
    SUM(Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit,
    COUNT(DISTINCT [Order ID]) AS Total_Orders
FROM sales;
"""

pd.read_sql(query, conn)

,Total_Revenue,Total_Profit,Total_Orders
0,2.297201e+06,286397.0217,5009


In [6]:
query = """
SELECT 
    Region,
    SUM(Sales) AS Total_Sales,
    SUM(Profit) AS Total_Profit,
    SUM(Profit) * 1.0 / SUM(Sales) AS Profit_Margin
FROM sales
GROUP BY Region
ORDER BY Total_Sales DESC;
"""

pd.read_sql(query, conn)

,Region,Total_Sales,Total_Profit,Profit_Margin
0,West,725457.8245,108418.4489,0.149448
1,East,678781.2400,91522.7800,0.134834
2,Central,501239.8908,39706.3625,0.079216
3,South,391721.9050,46749.4303,0.119343


In [7]:
query = """
SELECT 
    [Product Name],
    SUM(Sales) AS Total_Sales,
    SUM(Profit) AS Total_Profit
FROM sales
GROUP BY [Product Name]
ORDER BY Total_Profit DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Product Name,Total_Sales,Total_Profit
0,Canon imageCLASS 2200 Advanced Copier,61599.824,25199.9280
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384,7753.0390
2,Hewlett Packard LaserJet 3310 Copier,18839.686,6983.8836
3,Canon PC1060 Personal Laser Copier,11619.834,4570.9347
4,HP Designjet T520 Inkjet Large Format Printer ...,18374.895,4094.9766


In [9]:
query = """
SELECT 
    [Product Name],
    SUM(Profit) AS Total_Profit
FROM sales
GROUP BY [Product Name]
HAVING Total_Profit < 0
ORDER BY Total_Profit ASC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Product Name,Total_Profit
0,Cubify CubeX 3D Printer Double Head Print,-8879.9704
1,Lexmark MX611dhe Monochrome Laser Printer,-4589.9730
2,Cubify CubeX 3D Printer Triple Head Print,-3839.9904
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,-2876.1156
4,Bush Advantage Collection Racetrack Conference...,-1934.3976


In [10]:
query = """
SELECT 
    [Order Year],
    SUM(Sales) AS Total_Sales,
    LAG(SUM(Sales)) OVER (ORDER BY [Order Year]) AS Previous_Year_Sales,
    (SUM(Sales) - LAG(SUM(Sales)) OVER (ORDER BY [Order Year])) 
        * 1.0 / LAG(SUM(Sales)) OVER (ORDER BY [Order Year]) 
        AS YoY_Growth
FROM sales
GROUP BY [Order Year]
ORDER BY [Order Year];
"""

pd.read_sql(query, conn)

,Order Year,Total_Sales,Previous_Year_Sales,YoY_Growth
0,2014,484247.4981,NaN,NaN
1,2015,470532.5090,484247.4981,-0.028322
2,2016,609205.5980,470532.5090,0.294715
3,2017,733215.2552,609205.5980,0.203560
